# Export selected metric-learning embeddings

This stage consumes the persisted trial ledger, exactly one selected checkpoint
for each loss, and the baseline artifact. It writes only validation/test NPZ
embeddings plus schema-v1 handoff metadata; checkpoints remain local-only.

In [2]:
# Configuration (the only values an operator should need to review or edit).
from pathlib import Path
%pip install h5py

REPOSITORY_ROOT = next(
    root for root in (Path.cwd().resolve(), Path.cwd().resolve().parent)
    if (root / "GLOSSARY.md").exists() and (root / "face_name_metric_learning").exists()
)
MANIFEST_PATH = REPOSITORY_ROOT / "data" / "processed" / "manifest.csv"
HDF5_PATH = REPOSITORY_ROOT / "data" / "processed" / "faces.h5"
PACKAGE_DIR = REPOSITORY_ROOT / "artifacts" / "face_name_metric_learning"
TRIAL_RESULTS_PATH = PACKAGE_DIR / "trial_results.csv"
ENVIRONMENT_RECORD_PATH = PACKAGE_DIR / "setup_environment.json"
HANDOFF_MANIFEST_PATH = PACKAGE_DIR / "handoff_manifest.json"
EXPECTED_SPLIT_COUNTS = {"train": 45_078, "validation": 9_660, "test": 9_660}
EXPECTED_CLASS_COUNT = 100
EMBEDDING_DIMENSION = 512
BATCH_SIZE = 128
CHILD_SEED_POLICY = "SHA-256 named child streams by trial and concern"

import subprocess
import sys

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

import torch

from face_name_metric_learning import (
    build_handoff_manifest_from_environment,
    export_selected_embeddings,
    load_embedding_package,
    read_selected_trials,
    trial_checkpoint_path,
    validate_handoff_manifest,
    validate_manifest_hdf5,
    validate_trial_ledger,
    write_handoff_manifest,
)

for required in (MANIFEST_PATH, HDF5_PATH, TRIAL_RESULTS_PATH, ENVIRONMENT_RECORD_PATH):
    assert required.is_file(), f"Required persistent input is missing: {required}"
baseline_path = PACKAGE_DIR / "embeddings" / "baseline_iresnet100.npz"
assert baseline_path.is_file(), f"Baseline NPZ is missing: {baseline_path}"

data_record = validate_manifest_hdf5(
    MANIFEST_PATH,
    HDF5_PATH,
    expected_split_counts=EXPECTED_SPLIT_COUNTS,
    expected_class_count=EXPECTED_CLASS_COUNT,
)
ledger = validate_trial_ledger(TRIAL_RESULTS_PATH)
selected = read_selected_trials(TRIAL_RESULTS_PATH)
assert list(selected) == ["triplet_margin", "arcface", "supcon"]
assert len(selected) == 3 and len(ledger["rows"]) == 10
checkpoint_paths = {
    loss: trial_checkpoint_path(REPOSITORY_ROOT, trial)
    for loss, trial in selected.items()
}
missing = [path for path in checkpoint_paths.values() if not path.is_file()]
assert not missing, f"Selected checkpoints are missing: {missing}"
device = "cuda:0" if torch.cuda.is_available() else "cpu"

selected_outputs = export_selected_embeddings(
    repository_root=REPOSITORY_ROOT,
    manifest_path=MANIFEST_PATH,
    hdf5_path=HDF5_PATH,
    trial_results_path=TRIAL_RESULTS_PATH,
    package_dir=PACKAGE_DIR,
    batch_size=BATCH_SIZE,
    device=device,
    expected_dimension=EMBEDDING_DIMENSION,
)
assert list(selected_outputs) == [
    "triplet_margin_selected.npz",
    "arcface_selected.npz",
    "supcon_selected.npz",
]
for path in selected_outputs.values():
    load_embedding_package(
        path,
        manifest_path=MANIFEST_PATH,
        expected_dimension=EMBEDDING_DIMENSION,
    )
load_embedding_package(
    baseline_path,
    manifest_path=MANIFEST_PATH,
    expected_dimension=EMBEDDING_DIMENSION,
)


  Using cached h5py-3.16.0-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
Using cached h5py-3.16.0-cp310-cp310-manylinux_2_28_x86_64.whl (5.1 MB)
Note: you may need to restart the kernel to use updated packages.


ValueError: versions.h5py: must be a non-empty string

LooseLeaf: installed for this notebook: h5py==3.16.0. h5py was already imported: restart the kernel (Kernel → Restart Kernel) to use the new version — the variables are lost, the installed packages stay.


In [7]:
from pathlib import Path
import subprocess
import sys


REPOSITORY_ROOT = next(
    root for root in (Path.cwd().resolve(), Path.cwd().resolve().parent)
    if (root / "GLOSSARY.md").exists() and (root / "face_name_metric_learning").exists()
)
PACKAGE_DIR = REPOSITORY_ROOT / "artifacts" / "face_name_metric_learning"
MANIFEST_PATH = REPOSITORY_ROOT / "data" / "processed" / "manifest.csv"
HDF5_PATH = REPOSITORY_ROOT / "data" / "processed" / "faces.h5"
PACKAGE_DIR = REPOSITORY_ROOT / "artifacts" / "face_name_metric_learning"
TRIAL_RESULTS_PATH = PACKAGE_DIR / "trial_results.csv"
ENVIRONMENT_RECORD_PATH = PACKAGE_DIR / "setup_environment.json"
HANDOFF_MANIFEST_PATH = PACKAGE_DIR / "handoff_manifest.json"
EXPECTED_SPLIT_COUNTS = {"train": 45_078, "validation": 9_660, "test": 9_660}
EXPECTED_CLASS_COUNT = 100
EMBEDDING_DIMENSION = 512
CHILD_SEED_POLICY = "SHA-256 named child streams by trial and concern"

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from face_name_metric_learning import (
    build_handoff_manifest_from_environment,
    validate_handoff_manifest,
    validate_manifest_hdf5,
    write_handoff_manifest,
)

git_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPOSITORY_ROOT,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
manifest = build_handoff_manifest_from_environment(
    PACKAGE_DIR,
    environment_record_path=ENVIRONMENT_RECORD_PATH,
    manifest_path=MANIFEST_PATH,
    hdf5_path=HDF5_PATH,
    trial_results_path=TRIAL_RESULTS_PATH,
    child_seed_policy=CHILD_SEED_POLICY,
    split_counts=EXPECTED_SPLIT_COUNTS,
    class_count=EXPECTED_CLASS_COUNT,
    expected_dimension=EMBEDDING_DIMENSION,
    git_commit=git_commit,
)
write_handoff_manifest(HANDOFF_MANIFEST_PATH, manifest)
validated_manifest = validate_handoff_manifest(
    manifest,
    PACKAGE_DIR,
    manifest_path=MANIFEST_PATH,
    hdf5_path=HDF5_PATH,
    trial_results_path=TRIAL_RESULTS_PATH,
    expected_dimension=EMBEDDING_DIMENSION,
)
assert validated_manifest["schema_version"] == 1
assert len(validated_manifest["selected_trials"]) == 3
assert not any(PACKAGE_DIR.rglob("*.pt")), "Checkpoints must not enter the package"
print(HANDOFF_MANIFEST_PATH)

/nfs/home/aqu4eb/DS4002_Project3/artifacts/face_name_metric_learning/handoff_manifest.json
